# AIS Ship Trajectory Analysis with GeoPandas
This notebook queries AIS vessel positions from a SQLite database and visualizes ship trajectories on interactive maps using GeoPandas.

## 1. Set the MMSI of the ship you would like to work with:

In [ ]:
# 338839000 MMSI of the ship that goes in the carribean
# 369970910 MMSI of the ship that goes around in the san diego bay
# 338812000 MMSI of the ship that goes chillin in a Jacksonville Florida bay
# 366998000 MMSI of the ship going to Mobile Alabama
# 369970968 MMSI of the another ship chillin in the Jacksonville Florida bay
# 368011000 MMSI of the ship going down the coast of California
# 369970707 MMSI of the ship leaving the puerto rico bay
# 368869000 MMSI of the ship that went to go chill in the Jacksonville bay

# 7812e9 ICAO 
# adaae9 ICAO
# 4ba9e9 ICAO
# 8014e9 ICAO
# a25ce9 ICAO
# 8965e9 ICAO

# vehicle_type = "plane"
vehicle_type = "ship"

QUERY = "369970910"

## 2. Import Required Libraries

In [7]:
import sys
from pathlib import Path

# Add actint to path for imports
actint_path = Path("/home/daxtonb/JFN-Groundtruth-Tools/actint/src")
if str(actint_path) not in sys.path:
    sys.path.insert(0, str(actint_path))

from backend.mcp_servers.ais.helpers.vessel_query import get_vessel_position_history_helper
# from backend.mcp_servers.adsb.helpers.

## 3. Connect to Database and Query Ship Positions

In [11]:
# Query positions for a specific ship (configure as needed)
# For example: query_ais_positions({"MMSI": "123456789"}) or {"vessel_id": value}


try:
    if vehicle_type == "ship":
        SHIP_QUERY = {"MMSI": QUERY}  # Change MMSI or column name as needed
        vehicle_positions = get_vessel_position_history_helper(QUERY)
        print(f"Retrieved {len(vehicle_positions)} positions")
        id_index = 1
        timestamp_index = 2
        lat_index = 3
        lon_index = 4
    # elif vehicle_type == "plane":
    #     # PLANE_QUERY = {"icao": QUERY}
    #     # print(QUERY)
    #     # vehicle_positions = query_adsb_positions(PLANE_QUERY, sort=True)
    #     # print(f"Retrieved {len(vehicle_positions)} positions")
    #     # id_index = 1
    #     # timestamp_index = 7
    #     # lat_index = 8
    #     # lon_index = 9
        
    if vehicle_positions:
        print(f"Start position: {vehicle_positions[-1]}")
        print(f"End position: {vehicle_positions[0]}")
except Exception as e:
    print(f"Error querying database: {e}")
    vehicle_positions = []
    print(vehicle_positions)

Retrieved 776 positions
Start position: {'mmsi': 368869000, 'basedatetime': datetime.datetime(2023, 1, 1, 0, 0, 3, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')), 'lat': 30.17971, 'lon': -80.66435, 'sog': 11.9, 'cog': 49.7, 'heading': 511.0, 'status': 15, 'cargo': 35}
End position: {'mmsi': 368869000, 'basedatetime': datetime.datetime(2023, 1, 1, 18, 55, 4, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')), 'lat': 30.39707, 'lon': -81.40836, 'sog': 0.0, 'cog': 90.2, 'heading': 511.0, 'status': 15, 'cargo': 35}


## 4 Interactive map with markers (ipyleaflet) 

### This might take a while to load depending on how  many points you have

In [18]:
from ipyleaflet import Map, Marker, AwesomeIcon, Popup, Rectangle
from ipywidgets import HTML
import math

# Example structure for ais:
# vehicle_positions = [
#     [ship_id, timestamp, speed, lat, lon, ...],
#     ...
# ]
#
#
# Example structure for adsb:
# vehicle_positions = [
#     [acio, reg_num, type, desc, db_flags, ...],
#     ...
# ]


middle_point = vehicle_positions[len(vehicle_positions)//2]

map1 = Map(center=(middle_point['lat'], middle_point['lon']), zoom=10)

# first and last positions
last = vehicle_positions[0]
first = vehicle_positions[-1]

start_icon = AwesomeIcon(name="play", marker_color="green", icon_color="white")
end_icon = AwesomeIcon(name="flag", marker_color="red", icon_color="white")

def create_popup(position):
    ship_id = position['mmsi']
    timestamp = position['basedatetime']
    lat = position['lat']
    lon = position['lon']

    html = HTML(f"""
    <b>Ship MMSI:</b> {ship_id}<br>
    <b>Timestamp:</b> {timestamp}<br>
    <b>Latitude:</b> {lat}<br>
    <b>Longitude:</b> {lon}<br>
    """)

    return Popup(child=html, close_button=True, auto_close=False)

# Add first and last markers
start_marker = Marker(location=(first['lat'], first['lon']), icon=start_icon)
start_marker.popup = create_popup(first)
map1.add(start_marker)

end_marker = Marker(location=(last['lat'], last['lon']), icon=end_icon)
end_marker.popup = create_popup(last)
map1.add(end_marker)

# Add a clickable marker every 50 positions
for i, position in enumerate(vehicle_positions[1:-1], start=1):
    if i % 100 == 0:
        icon = AwesomeIcon(name="circle", marker_color="blue", icon_color="white")
        marker = Marker(location=(position['lat'], position['lon']), icon=icon)
        marker.popup = create_popup(position)
        map1.add(marker)
    elif i%4 == 0:
        marker = Marker(location=(position['lat'], position['lon']))
        map1.add(marker)

map1

Map(center=[30.28306, -80.77139], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', …

## 5. Filter and Analyze Multiple Ship Trajectories

## Points used to determine where the ship is going

In [ ]:
NUMBER_DETECTIONS=300

if len(vehicle_positions) > NUMBER_DETECTIONS:
    num_det = NUMBER_DETECTIONS
else:
    num_det = len(vehicle_positions)

map2 = Map(center=(middle_point[lat_index], middle_point[lon_index]), zoom=10)

# first and last positions
first = vehicle_positions[num_det-1]
last = vehicle_positions[0]

start_icon = AwesomeIcon(name="play", marker_color="green", icon_color="white")
end_icon = AwesomeIcon(name="flag", marker_color="red", icon_color="white")

# Add first and last markers
start_marker = Marker(location=(first[lat_index], first[lon_index]), icon=start_icon)
start_marker.popup = create_popup(first)
map2.add(start_marker)

end_marker = Marker(location=(last[lat_index], last[lon_index]), icon=end_icon)
end_marker.popup = create_popup(last)
map2.add(end_marker)

# Add a clickable marker every 50 positions
print(len(vehicle_positions[1:num_det-1]))
for i, position in enumerate(vehicle_positions[1:num_det-1], start=1):
    if i % 100 == 0:
        icon = AwesomeIcon(name="circle", marker_color="blue", icon_color="white")
        marker = Marker(location=(position[lat_index], position[lon_index]), icon=icon)
        marker.popup = create_popup(position)
        map2.add(marker)
    else:
        marker = Marker(location=(position[lat_index], position[lon_index]))
        map2.add(marker)

map2

40


Map(center=[22.228271, 113.47254], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

## A couple of helper functions

In [ ]:
def vectorize(lat1, lon1, lat2, lon2):
    lat = lat2-lat1
    lon = lon2-lon1
    return (lat, lon)


def within_angle(a, b, tolerance=15):
    diff = abs((a - b + 180) % 360 - 180)
    return diff <= tolerance

## Code to determine what direction the ship is gonig

In [ ]:
VECTOR_DISTANCE_RATIO = 0.9
DEGREE_THRESHOLD= 15
 #This is the number of detections used to calculate where a ship is going.
from actint.tools.utils.important_locations import *
from actint.data_processing.query_database import query_ais_positions
from datetime import datetime, timedelta
from actint.tools.utils.distance_calculation import calculate_bearing, haversine_distance_nm
from actint.tools.lat_lon_context import identify_maritime_region

# everything is in one function here.
def calculate_vector_and_distance_sum(ship_mmsi: str, vehicle_type: str, number_detections=300, tracking_time=timedelta(hours=1)):
    if vehicle_type == "ship":
        positions = query_ais_positions({"mmsi": ship_mmsi}, sort=True)
        id_index = 1
        timestamp_index = 2
        lat_index = 3
        lon_index = 4
    elif vehicle_type == "plane":
        positions = query_adsb_positions({"icao": ship_mmsi}, sort=True)
        id_index = 1
        timestamp_index = 7
        lat_index = 8
        lon_index = 9

    if number_detections > len(positions):
        number_detections = len(positions)
    

    position1 = positions[number_detections-1]        #This is 300 or whatever the number_detections is away from the most rescent position
    print(position1)

    rescent_reversed_positions = reversed(positions[0:number_detections-1])
    total_vector = [0.0, 0.0]
    total_distance = 0

    for position2 in rescent_reversed_positions:
        print(position2[lat_index], position2[lon_index], position2[timestamp_index])
        latlng = vectorize(position1[lat_index], position1[lon_index], position2[lat_index], position2[lon_index])
        total_vector[0] += latlng[0]
        total_vector[1] += latlng[1]
        total_distance += math.hypot(latlng[0], latlng[1])
        position1 = position2
    
    

    vector_distance_ratio = math.hypot(total_vector[0], total_vector[1])/total_distance
    print("Vector distance ratio", vector_distance_ratio)
    if(vector_distance_ratio > VECTOR_DISTANCE_RATIO):                                                       #Can use this to describe if the ship is going fast or slow
        print("The ship is going toward something")
        return (total_vector, total_distance)
    else:
        print("The ship is doing wierd stuff acting like a reet.")
        return (total_vector, total_distance)
    

        
(total_vector, total_distance) = calculate_vector_and_distance_sum(QUERY, vehicle_type, 300)

print(total_vector)

(1, '7812e9', 'B-1376', 'B738', 'BOEING 737-800', 0, 0, 1745119356.06, 22.464294, 113.754679, 7375, 259.9, 228.3, 3, 2624, 'adsb_icao', 7775, 2656, '248', '-2.1', 1, 1, 0, 0, '2026-04-09 16:46:23')
22.46057 113.750153 1745119360.69
22.458662 113.747813 1745119363.15
22.454544 113.742803 1745119368.15
22.450096 113.737335 1745119373.53
22.442505 113.728222 1745119382.67
22.437527 113.722229 1745119388.82
22.435758 113.720144 1745119391.06
22.431522 113.715159 1745119395.9
22.429939 113.713226 1745119397.86
22.422501 113.704501 1745119406.64
22.418115 113.69929 1745119411.65
22.41449 113.695013 1745119415.83
22.348519 113.616689 1745119485.31
22.343398 113.610586 1745119490.31
22.333466 113.598833 1745119499.87
22.327338 113.591614 1745119505.7
22.246942 113.496806 1745119582.64
22.245026 113.494612 1745119584.6
22.232045 113.478241 1745119597.44
22.228271 113.47254 1745119601.65
22.223373 113.46445 1745119607.49
22.21794 113.45403 1745119614.59
22.214961 113.447673 1745119618.96
22.2107

In [ ]:
current_position = vehicle_positions[0]

print(current_position)
map3 = Map(center=(current_position[3], current_position[4]), zoom=10)

# Add first and last markers
current_pos_marker = Marker(location=(current_position[3], current_position[4]))
current_pos_marker.popup = create_popup(current_position)
map3.add(current_pos_marker)

(dy, dx) = total_vector

start_lat = float(current_position[lat_index])
start_lon = float(current_position[lon_index])

end_lat = start_lat + dy
end_lon = start_lon + dx

from ipyleaflet import Polyline

vector_line = Polyline(
    locations=[(start_lat, start_lon), (end_lat, end_lon)],
    color="red",
    weight=3
)

map3.add(vector_line)

map3

(42, '7812e9', 'B-1376', 'B738', 'BOEING 737-800', 0, 0, 1745146621.31, 22.706459, 113.728689, 13400, None, None, 1, None, 'adsb_icao', None, None, None, None, 1, 0, 0, 0, '2026-04-09 16:46:23')


Map(center=['B738', 'BOEING 737-800'], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

## Code to determine how long a ship has been staying at a specific place (uses information from the previous code)

In [ ]:
from geographiclib.geodesic import Geodesic

DISTANCE_THRESHOLD = 500 # Meters from where the ship has been
TIME_THRESHOLD = timedelta(minutes=20) # A ship is considered to be staying still if it has been within DISTANCE_THRESHOLD for TIME_THRESHOLD minutes
NUMBER_DETECTIONS_TRESHOLD = 5

ship_staying_still = False

geod = Geodesic.WGS84
latest_position = vehicle_positions[0]

latest_time = datetime.strptime(vehicle_positions[0][2], '%Y-%m-%dT%H:%M:%S.%f')

number_detections_still = 0
time_stayed_still = 0

for (num, ship_position) in enumerate(vehicle_positions[1:]):
    path = geod.Inverse(latest_position[3], latest_position[4], ship_position[3], ship_position[4])
    distance = path['s12']
    if(distance > DISTANCE_THRESHOLD):
        break
    if(num > NUMBER_DETECTIONS_TRESHOLD and not ship_staying_still):
        if (TIME_THRESHOLD < latest_time - datetime.strptime(ship_position[2], '%Y-%m-%dT%H:%M:%S.%f')):
            ship_staying_still = True #at this point all the conditions for the ship to be staying still are true, it hasn't moved more than DISTANCE_THRESHOLD, has been still for NUMBER_DETECTIONS_TRESHOLD, for at least TIME_THRESHOLD
    time_stayed_still = latest_time - datetime.strptime(ship_position[2], '%Y-%m-%dT%H:%M:%S.%f')
    number_detections_still += 1

if(ship_staying_still):
    total_seconds = int(time_stayed_still.total_seconds())
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    print(f"The ship has been still for {number_detections_still} detections and for a total time of {hours}h {minutes}m {seconds}s")
else:
    print("The ship is not currently staying still.")

The ship is not currently staying still.
